# 🧠 การเพิ่มประสิทธิภาพแบบ RMSProp: พิกัดที่ปรับเปลี่ยนได้ (Adaptive Coordinates)

ยินดีต้อนรับสู่สมุดบันทึกคำอธิบายเชิงปฏิบัติสำหรับ **RMSProp Optimization**! ในสมุดบันทึกนี้ เราจะ:
1. อธิบายข้อจำกัดของ AdaGrad (การลดลงของอัตราการเรียนรู้จนเกือบเป็นศูนย์) และวิธีที่ค่าเฉลี่ยสลายตัวแบบเอ็กซ์โพเนนเชียล (exponentially decaying average) ของ RMSProp เข้ามาแก้ปัญหานี้
2. อิมพลีเมนต์ **Vanilla Gradient Descent**, **AdaGrad** และ **RMSProp** จากศูนย์
3. ติดตามและเปรียบเทียบเส้นทางการลู่เข้าสำหรับเพิ่มประสิทธิภาพบนฟังก์ชันต้นทุนหุบเหว 2 มิติที่ลาดชัน:
   $$f(x, y) = 0.5x^2 + 10y^2$$
4. แสดงภาพเส้นทางของพวกมันบนแผนที่เส้นชั้นความสูงแบบ 2 มิติ เพื่อสังเกตวิธีที่ AdaGrad หยุดการเคลื่อนไหว (freezes) ก่อนเวลาอันควร และวิธีที่ RMSProp ปรับอัตราการเรียนรู้ของแต่ละพิกัดอย่างมีพลวัต
5. พล็อตเส้นกราฟการลดลงของลอสเพื่อเปรียบเทียบความเร็วในการลู่เข้า
6. เชื่อมโยงแนวคิดอัตราการเรียนรู้แบบปรับเปลี่ยนได้ (adaptive learning rates) กับการฝึกโครงข่ายประสาทเทียมเชิงลึก (เช่น การฝึกในแต่ละชั้นของโมเดล YOLO)

มาเริ่มต้นด้วยการนำเข้าไลบรารีที่จำเป็นกันเลย

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Set seed for reproducibility
np.random.seed(42)

## 1. การกำหนดฟังก์ชันหุบเหว (Ravine Function) และเกรเดียนต์

ฟังก์ชันต้นทุนของเราแสดงถึงหุบเขาลึก:
$$f(x, y) = 0.5x^2 + 10y^2$$

เกรเดียนต์:
$$\frac{\partial f}{\partial x} = x, \quad \frac{\partial f}{\partial y} = 20y$$

In [ ]:
def cost_ravine(x, y):
    return 0.5 * x**2 + 10.0 * y**2

def grad_ravine(x, y):
    return np.array([x, 20.0 * y])

## 2. การอิมพลีเมนต์ตัวปรับค่า (Optimizers) จากศูนย์

ลองเขียนลูปการเพิ่มประสิทธิภาพทั้งสามแบบ:
1.  **Vanilla GD:** $\mathbf{w}_{t+1} = \mathbf{w}_t - \alpha \nabla J(\mathbf{w}_t)$
2.  **AdaGrad:** ใช้ผลรวมสะสมของเกรเดียนต์ยกกำลังสองในตัวส่วน
3.  **RMSProp:** ใช้ค่าเฉลี่ยวิ่งแบบสลายตัวเป็นเอ็กซ์โพเนนเชียลของเกรเดียนต์ยกกำลังสองในตัวส่วน

In [ ]:
def optimize_vanilla(start_pos, lr=0.15, epochs=60):
    pos = np.array(start_pos, dtype=float)
    history = [pos.copy()]
    for _ in range(epochs):
        grad = grad_ravine(pos[0], pos[1])
        pos -= lr * grad
        history.append(pos.copy())
    return np.array(history)

def optimize_adagrad(start_pos, lr=0.5, eps=1e-8, epochs=60):
    pos = np.array(start_pos, dtype=float)
    s = np.zeros(2)
    history = [pos.copy()]
    for _ in range(epochs):
        grad = grad_ravine(pos[0], pos[1])
        s += grad ** 2
        pos -= (lr / (np.sqrt(s) + eps)) * grad
        history.append(pos.copy())
    return np.array(history)

def optimize_rmsprop(start_pos, lr=0.15, beta=0.9, eps=1e-8, epochs=60):
    pos = np.array(start_pos, dtype=float)
    v = np.zeros(2)
    history = [pos.copy()]
    for _ in range(epochs):
        grad = grad_ravine(pos[0], pos[1])
        v = beta * v + (1.0 - beta) * (grad ** 2)
        pos -= (lr / (np.sqrt(v) + eps)) * grad
        history.append(pos.copy())
    return np.array(history)

# Run optimizations starting at (8.0, 4.0)
start = [8.0, 4.0]
path_vanilla = optimize_vanilla(start, lr=0.08)
path_adagrad = optimize_adagrad(start, lr=0.8)
path_rmsprop = optimize_rmsprop(start, lr=0.15, beta=0.9)

## 3. การแสดงภาพเส้นทางการลู่เข้าบนแผนที่เส้นชั้นความสูง (Contour Map)

มาสร้างกริดเส้นชั้นความสูงแบบ 2 มิติและพล็อตเส้นทางการลู่เข้ากัน

In [ ]:
x = np.linspace(-10, 10, 150)
y = np.linspace(-5, 5, 150)
X, Y = np.meshgrid(x, y)
Z = cost_ravine(X, Y)

plt.figure(figsize=(12, 8))
contours = plt.contour(X, Y, Z, levels=30, cmap='viridis')
plt.clabel(contours, inline=1, fontsize=8)

# Plot paths
plt.plot(path_vanilla[:, 0], path_vanilla[:, 1], color='red', marker='o', alpha=0.8, linewidth=1.5, label='Vanilla GD')
plt.plot(path_adagrad[:, 0], path_adagrad[:, 1], color='purple', marker='^', linewidth=2, label='AdaGrad (Stalls early)')
plt.plot(path_rmsprop[:, 0], path_rmsprop[:, 1], color='cyan', marker='s', linewidth=2.5, label='RMSProp (Glides to minimum)')

plt.scatter(0, 0, color='gold', s=150, marker='*', zorder=5, label='Minimum (0,0)')
plt.xlabel('x')
plt.ylabel('y')
plt.xlim(-10, 10)
plt.ylim(-5, 5)
plt.title('AdaGrad Stalling vs. RMSProp Adaptive Adjustments')
plt.legend()
plt.show()

ดูพล็อตสิ!
-   **Vanilla GD (สีแดง):** แกว่งไปมาอย่างรุนแรงเนื่องจากเกรเดียนต์ของแกน y ลาดชันมาก ส่งผลให้เคลื่อนที่ช้ามากไปยังจุด $x=0$
-   **AdaGrad (สีม่วง):** เนื่องจากเกรเดียนต์ในช่วงแรกมีค่าใหญ่มาก ผลรวมสะสมในประวัติ $s$ จึงพุ่งสูงขึ้นอย่างรวดเร็ว ส่งผลให้อัตราการเรียนรู้ถูกหารจนเกือบเป็นศูนย์ ทำให้มันหยุดเคลื่อนไหว (stalls) ห่างจากจุดต่ำสุดรวมมาก
-   **RMSProp (สีฟ้าคราม):** การเฉลี่ยแบบสลายตัวของเกรเดียนต์ยกกำลังสองจะช่วยป้องกันไม่ให้ตัวส่วนเพิ่มขึ้นแบบระเบิด มันจะช่วยลดการแกว่งในแนวตั้ง (แกน $y$) ในขณะที่ปรับขนาดเพิ่มระยะการอัปเดตในแนวนอน (แกน $x$) ทำให้ลื่นไหลตรงไปยังจุดต่ำสุดได้สำเร็จ

## 4. การเปรียบเทียบความเร็วในการลู่เข้า

ลองพล็อตเส้นกราฟการลดลงของต้นทุนกัน

In [ ]:
cost_vanilla = [cost_ravine(p[0], p[1]) for p in path_vanilla]
cost_adagrad = [cost_ravine(p[0], p[1]) for p in path_adagrad]
cost_rmsprop = [cost_ravine(p[0], p[1]) for p in path_rmsprop]

plt.figure(figsize=(10, 5))
plt.plot(cost_vanilla, color='red', label='Vanilla GD')
plt.plot(cost_adagrad, color='purple', label='AdaGrad')
plt.plot(cost_rmsprop, color='cyan', label='RMSProp')
plt.yscale('log')
plt.xlabel('Steps')
plt.ylabel('Log Cost')
plt.title('Cost Convergence Comparison (Log Scale)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.show()

## 💡 ความเชื่อมโยงกับ YOLO และการเรียนรู้เชิงลึก (Deep Learning)
*   **ปัญหาเกรเดียนต์หายในแต่ละชั้น (Vanishing Gradients across layers):** ในโครงข่ายเชิงลึก (เช่น โครงสร้าง CNN Backbone ของ YOLO) เลเยอร์คอนโวลูชันช่วงแรกๆ จะอยู่ห่างจากค่าลอสเอาต์พุตมาก เกรเดียนต์จึงมีขนาดเล็กมาก ในทางตรงกันข้าม เลเยอร์ส่วนหัวสำหรับการจำแนกประเภทและการทำนายกรอบ (final classification and regression heads) จะมีเกรเดียนต์ที่ใหญ่กว่าอย่างเห็นได้ชัด
*   **การปรับขนาดพารามิเตอร์อย่างอิสระ (Adaptive Scaling):** หากเราใช้อัตราการเรียนรู้ร่วมกันเพียงอัตราเดียวสำหรับทั้งโมเดล เราอาจจะอัปเดตเลยค่าที่เหมาะสมที่สุดของเลเยอร์ส่วนท้าย หรือไม่ก็ไม่สามารถอัปเดตพารามิเตอร์ของเลเยอร์แรกๆ ได้ ตัวปรับแต่งแบบปรับเปลี่ยนอัตโนมัติเช่น RMSProp (และตัวที่พัฒนาต่อยอดอย่าง Adam) จะปรับขนาดพารามิเตอร์แบบแยกเฉพาะจากกัน ทำให้แต่ละเลเยอร์สามารถเรียนรู้ได้ด้วยความเร็วที่เหมาะสมที่สุดของตัวเอง!